In [1]:
from langchain_community.document_loaders import PyPDFLoader

c:\Users\USER\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
## Vector Embedding And Vector Store
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


In [3]:
loader=PyPDFLoader(r'doc\attention.pdf')
docs=loader.load()
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'doc\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nl

In [4]:
print("number of pages :",len(docs))

number of pages : 15


In [5]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 52 sub-documents.


In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


3. Vector stores

vector_store.add_documents(documents=all_splits)
LangChain automatically does:
Document Text
      ↓
embed_documents()
      ↓
Embedding Vector
      ↓
Store in Vector Database

In [7]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(all_splits , embeddings)
#vectorstore.add_documents(all_splits) # -- vectorstore.add_documents: Use this to append more documents to an already existing vectorstore
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2})

In [9]:
vectorstore

In [ ]:
#You can access the internal mapping of FAISS indices to document IDs via the
print(vectorstore.index_to_docstore_id)

{0: 'a1992de1-7489-4747-8d36-530d32e4ee3e', 1: '5b22fdc9-7d13-43e8-8e2c-759ba17cd667', 2: '5b25b2f6-a172-44c5-ab6c-db48c0826794', 3: '76551b72-96ed-4668-8d85-7f9404476808', 4: '10eb6d47-e8be-479b-84a1-58f009fbed4a', 5: '6c75e5c4-4169-43d6-b57d-7639fb6bafa7', 6: '19d6a660-c62b-4d45-9569-5b6af912309b', 7: 'e751d1c2-3fe6-4088-85a8-31daf3cda430', 8: 'b58d85b1-74ed-4d85-96ba-ebcff5c2a6ed', 9: '75ee53ba-5c41-4933-876e-4b4cef9a8784', 10: 'b4b94d97-d4ac-49cf-ae0a-f5e803a409d4', 11: '4d7be7a8-09e8-41e7-9b84-8d9e53e6781a', 12: 'e447b53a-32eb-44b2-bfe9-df0e846a2ef7', 13: 'ec3bcb97-c9c6-499f-b8c5-893716763ccb', 14: '07d0c240-3d4d-45a4-9211-8c694e41ee10', 15: 'e887f1d3-5d3e-4c86-80f8-a94f2474710d', 16: 'b4f41d60-95e8-43ba-a8bb-1420b1ff1f9e', 17: '7281ff6b-c063-4666-9858-c8468baad55f', 18: '9e7cda47-0888-46cf-b266-0eb593190edc', 19: '325597d5-ab1f-41f7-aa30-c28cbc3786b8', 20: '5ced27d9-d679-480e-93cd-6c487f11983a', 21: 'de7a67c6-bfef-4c3b-b9d6-cd6f36453945', 22: '976eae94-7f4e-44db-aa2c-5bdf4a0f1cb9

To get a specific Document by ID

In [16]:
doc_id = vectorstore.index_to_docstore_id[0] # Get the ID for the first vector
document = vectorstore.docstore.search(doc_id)
print(document)

page_content='Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolution

retrive from vector store

In [15]:
query = "convolutional neural networks"
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retriever.invoke(query)

[Document(id='60b0a6f8-820e-4d92-a1ce-7af2c7cac152', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'doc\\attention.pdf', 'total_pages': 15, 'page': 6, 'page_label': '7', 'start_index': 785}, page_content='or O(logk(n)) in the case of dilated convolutions [ 18], increasing the length of the longest paths\nbetween any two positions in the network. Convolutional layers are generally more expensive than\nrecurrent layers, by a factor of k. Separable convolutions [ 6], however, decrease the complexity\nconsiderably, to O(k · n · d + n · d2). Even with k = n, however, the complexity of a separable\nconvolution is equal to the combination of a self-attention layer and a point-wise feed-f

In [17]:
results = vectorstore.similarity_search(
    query=query,
    k=3
)

print(results[0])

page_content='or O(logk(n)) in the case of dilated convolutions [ 18], increasing the length of the longest paths
between any two positions in the network. Convolutional layers are generally more expensive than
recurrent layers, by a factor of k. Separable convolutions [ 6], however, decrease the complexity
considerably, to O(k · n · d + n · d2). Even with k = n, however, the complexity of a separable
convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,
the approach we take in our model.
As side benefit, self-attention could yield more interpretable models. We inspect attention distributions
from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic
and semantic structure of the sentences.
5 Training
This section describes the training regime for our models.
5.1 Training Data and Batching' meta

In [23]:
print(results)

[Document(id='23c32a75-75a2-4d7e-b9cf-42b15d840e81', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'doc\\attention.pdf', 'total_pages': 15, 'page': 1, 'page_label': '2', 'start_index': 2341}, page_content='in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes\nit more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an at

In [18]:
results = vectorstore.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 1.7088921070098877

page_content='efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating
our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.
31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
arXiv:1706.03762v7  [cs.CL]  2 Aug 2023' metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'doc\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'start_index': 2408}


In [20]:
results[1]

(Document(id='722716a9-1b5c-409a-9552-5fc47085a99f', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'doc\\attention.pdf', 'total_pages': 15, 'page': 8, 'page_label': '9', 'start_index': 765}, page_content='(E) positional embedding instead of sinusoids 4.92 25.7\nbig 6 1024 4096 16 0.3 300K 4.33 26.4 213\ndevelopment set, newstest2013. We used beam search as described in the previous section, but no\ncheckpoint averaging. We present these results in Table 3.\nIn Table 3 rows (A), we vary the number of attention heads and the attention key and value dimensions,\nkeeping the amount of computation constant, as described in Section 3.2.2. While single-head\nattention is 0.9 BLEU worse t